# TechOps Intelligence Platform
## Notebook 08 — LangGraph Agent Pipeline

**Company:** FinTechFlow — B2B Payment Processor  
**Goal:** Build 4-agent LangGraph pipeline for incident intelligence

### Agent Pipeline
1. Triage Agent     → classify severity + category
2. Diagnosis Agent  → retrieve root cause from knowledge base  
3. Resolution Agent → fetch runbook steps + fix procedure
4. Comms Agent      → draft stakeholder message + postmortem

### LLM
- Development : Qwen2.5 7B via Ollama (local, free)
- Production  : Claude API (swap one line)

In [2]:
import os
import json
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
load_dotenv()
os.environ["LANGCHAIN_TRACING_V2"]          = "false"
os.environ["LANGCHAIN_CALLBACKS_BACKGROUND"] = "false"
os.environ["LANGCHAIN_API_KEY"]              = ""
os.environ["LANGSMITH_API_KEY"] = ""

os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from pathlib import Path
from typing import TypedDict, Optional, List
import numpy as np

from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph, END
import chromadb

PROJECT_ROOT = Path("C:/Users/sudha/techops-intelligence")
os.chdir(PROJECT_ROOT)

import sys
sys.path.insert(0, str(PROJECT_ROOT))

EMBEDDINGS = PROJECT_ROOT / "data/embeddings"

print("Setup complete")
print(f"Project root: {PROJECT_ROOT}")

c:\Users\sudha\techops-intelligence\venv\Lib\site-packages\langgraph\checkpoint\base\__init__.py:21: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Setup complete
Project root: C:\Users\sudha\techops-intelligence


In [3]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from src.agents.triage_classifier import classify_incident
from src.retrieval.retriever import retrieve, format_results

# Embedding model
print("Loading embedding model...")
embedding_model = SentenceTransformer(
    'sentence-transformers/all-mpnet-base-v2',
    device='cpu'
)
print("Embedding model ready")

# Cross encoder
print("Loading cross encoder...")
cross_encoder = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-6-v2',
    device='cpu'
)
print("Cross encoder ready")

# LLM — Qwen2.5 via Ollama (new import)
print("Loading Ollama LLM...")
from langchain_ollama import OllamaLLM

llm = OllamaLLM(
    model       = "qwen2.5:7b",
    base_url    = "http://localhost:11434",
    temperature = 0.1,
    num_ctx     = 4096
)

# Quick test
test_response = llm.invoke("Reply with exactly: READY")
print(f"LLM response: {test_response.strip()}")


# ChromaDB
chroma_path = str(EMBEDDINGS / "chroma_db")
client      = chromadb.PersistentClient(path=chroma_path)

collections = {}
for name in ['incidents', 'postmortems',
             'playbooks', 'knowledge_base', 'logs']:
    try:
        collections[name] = client.get_collection(name)
        print(f"  {name:20}: {collections[name].count():,} docs")
    except Exception as e:
        print(f"  {name:20}: not found")

print("\nAll resources loaded")


Loading embedding model...
Embedding model ready
Loading cross encoder...
Cross encoder ready
Loading Ollama LLM...
LLM response: READY


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


  incidents           : 2,000 docs
  postmortems         : 175 docs
  playbooks           : 100 docs
  knowledge_base      : 11,975 docs
  logs                : 1,120 docs

All resources loaded


## 1. Agent State
Shared state passed between all 4 agents
Each agent reads from and writes to this state

In [4]:
class AgentState(TypedDict):
    # Input
    incident_text     : str

    # Triage Agent output
    severity          : str
    category          : str
    severity_confidence: float
    category_confidence: float
    severity_source   : str

    # Diagnosis Agent output
    retrieved_incidents : List[str]
    retrieved_postmortems: List[str]
    root_cause          : str
    diagnosis_confidence: float

    # Resolution Agent output
    retrieved_runbooks  : List[str]
    resolution_steps    : List[str]
    estimated_time_mins : int
    escalation_needed   : bool

    # Comms Agent output
    stakeholder_message : str
    postmortem_draft    : str

    # Pipeline metadata
    agent_latencies     : dict
    error               : Optional[str]


print("AgentState defined")
print("Fields:", list(AgentState.__annotations__.keys()))

AgentState defined
Fields: ['incident_text', 'severity', 'category', 'severity_confidence', 'category_confidence', 'severity_source', 'retrieved_incidents', 'retrieved_postmortems', 'root_cause', 'diagnosis_confidence', 'retrieved_runbooks', 'resolution_steps', 'estimated_time_mins', 'escalation_needed', 'stakeholder_message', 'postmortem_draft', 'agent_latencies', 'error']


## 2. Triage Agent
Classifies incident severity and category
Uses hybrid DistilBERT + rule-based classifier
Fast — no LLM call needed

In [5]:
import time

def triage_agent(state: AgentState) -> AgentState:
    """
    Agent 1: Triage
    Input : raw incident text
    Output: severity (P1-P3), category (8 types),
            confidence scores
    Uses  : hybrid DistilBERT + rule-based classifier
    LLM   : NO — fast local inference
    """
    start = time.time()
    print(f"\n[TRIAGE] Processing incident...")

    try:
        result = classify_incident(state['incident_text'])

        state['severity']           = result['severity']
        state['category']           = result['category']
        state['severity_confidence'] = result['severity_confidence']
        state['category_confidence'] = result['category_confidence']
        state['severity_source']    = result.get(
            'severity_source', 'distilbert'
        )

        latency = round(time.time() - start, 3)
        state['agent_latencies'] = state.get(
            'agent_latencies', {}
        )
        state['agent_latencies']['triage'] = latency

        print(f"[TRIAGE] Severity : {result['severity']} "
              f"({result['severity_confidence']:.2f}) "
              f"via {state['severity_source']}")
        print(f"[TRIAGE] Category : {result['category']} "
              f"({result['category_confidence']:.2f})")
        print(f"[TRIAGE] Latency  : {latency}s")

    except Exception as e:
        state['severity']  = 'P2'
        state['category']  = 'application'
        state['error']     = f"Triage error: {str(e)}"
        print(f"[TRIAGE] Error: {e} — defaulting to P2/application")

    return state


# Test triage agent
test_state = AgentState(
    incident_text      = "PostgreSQL connection refused ECONNREFUSED port 5432 payment-service max_connections exhausted",
    severity           = "",
    category           = "",
    severity_confidence = 0.0,
    category_confidence = 0.0,
    severity_source    = "",
    retrieved_incidents  = [],
    retrieved_postmortems= [],
    root_cause           = "",
    diagnosis_confidence = 0.0,
    retrieved_runbooks   = [],
    resolution_steps     = [],
    estimated_time_mins  = 0,
    escalation_needed    = False,
    stakeholder_message  = "",
    postmortem_draft     = "",
    agent_latencies      = {},
    error                = None
)

result_state = triage_agent(test_state)
print(f"\nTriage test passed: {result_state['severity']} / {result_state['category']}")


[TRIAGE] Processing incident...
[TRIAGE] Severity : P1 (0.85) via rule_based
[TRIAGE] Category : database (0.96)
[TRIAGE] Latency  : 0.265s

Triage test passed: P1 / database


## 3. Diagnosis Agent
Retrieves similar incidents and postmortems
Uses hybrid retrieval (BM25 + semantic + reranking)
LLM generates root cause hypothesis from retrieved context

In [6]:
DIAGNOSIS_PROMPT = PromptTemplate.from_template("""
You are a senior SRE at FinTechFlow diagnosing a production incident.

INCIDENT:
{incident_text}

SEVERITY: {severity} | CATEGORY: {category}

SIMILAR PAST INCIDENTS:
{similar_incidents}

RELEVANT POSTMORTEMS:
{postmortems}

Based on the incident details and historical context above,
provide a concise technical diagnosis.

Respond in this exact format:
ROOT_CAUSE: [one sentence technical root cause]
CONFIDENCE: [HIGH/MEDIUM/LOW]
CONTRIBUTING_FACTORS: [comma separated list of 2-3 factors]
IMMEDIATE_ACTION: [single most important first step]
""")


def diagnosis_agent(state: AgentState) -> AgentState:
    """
    Agent 2: Diagnosis
    Input : triage output + incident text
    Output: root cause, confidence, contributing factors
    Uses  : hybrid retrieval from incidents + postmortems
    LLM   : YES — Qwen2.5 7B via Ollama
    """
    start = time.time()
    print(f"\n[DIAGNOSIS] Retrieving similar incidents...")

    try:
        # Retrieve similar incidents
        incident_results = retrieve(
            query            = state['incident_text'],
            collection_names = ['incidents'],
            top_k_fetch      = 10,
            top_k_return     = 3,
            use_reranking    = True
        )

        # Retrieve relevant postmortems
        pm_query = (
            f"{state['category']} incident "
            f"{state['incident_text'][:200]}"
        )
        postmortem_results = retrieve(
            query            = pm_query,
            collection_names = ['postmortems'],
            top_k_fetch      = 10,
            top_k_return     = 3,
            use_reranking    = True
        )

        # Format retrieved content
        similar_incidents = "\n\n".join([
            f"[{i+1}] {r['text'][:300]}"
            for i, r in enumerate(incident_results)
        ]) or "No similar incidents found"

        postmortems_text = "\n\n".join([
            f"[{i+1}] {r['text'][:400]}"
            for i, r in enumerate(postmortem_results)
        ]) or "No postmortems found"

        state['retrieved_incidents']   = [
            r['text'][:200] for r in incident_results
        ]
        state['retrieved_postmortems'] = [
            r['text'][:200] for r in postmortem_results
        ]

        print(f"[DIAGNOSIS] Retrieved {len(incident_results)} incidents, "
              f"{len(postmortem_results)} postmortems")
        print(f"[DIAGNOSIS] Generating diagnosis with LLM...")

        # Generate diagnosis with LLM
        prompt = DIAGNOSIS_PROMPT.format(
            incident_text    = state['incident_text'],
            severity         = state['severity'],
            category         = state['category'],
            similar_incidents = similar_incidents,
            postmortems      = postmortems_text
        )

        response = llm.invoke(prompt)

        # Parse LLM response
        root_cause   = "Unable to determine root cause"
        confidence   = "MEDIUM"

        for line in response.split('\n'):
            line = line.strip()
            if line.startswith('ROOT_CAUSE:'):
                root_cause = line.replace('ROOT_CAUSE:', '').strip()
            elif line.startswith('CONFIDENCE:'):
                confidence = line.replace('CONFIDENCE:', '').strip()

        conf_map = {'HIGH': 0.9, 'MEDIUM': 0.7, 'LOW': 0.5}

        state['root_cause']           = root_cause
        state['diagnosis_confidence'] = conf_map.get(
            confidence.upper(), 0.7
        )

        latency = round(time.time() - start, 3)
        state['agent_latencies']['diagnosis'] = latency

        print(f"[DIAGNOSIS] Root cause: {root_cause[:100]}")
        print(f"[DIAGNOSIS] Confidence: {confidence}")
        print(f"[DIAGNOSIS] Latency   : {latency}s")

    except Exception as e:
        state['root_cause']           = f"Diagnosis failed: {str(e)}"
        state['diagnosis_confidence'] = 0.0
        state['error']                = f"Diagnosis error: {str(e)}"
        print(f"[DIAGNOSIS] Error: {e}")

    return state


# Test diagnosis agent
print("Testing diagnosis agent...")
result_state = diagnosis_agent(result_state)
print(f"\nDiagnosis test passed")
print(f"Root cause: {result_state['root_cause'][:150]}")

Testing diagnosis agent...

[DIAGNOSIS] Retrieving similar incidents...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[DIAGNOSIS] Retrieved 3 incidents, 3 postmortems
[DIAGNOSIS] Generating diagnosis with LLM...
[DIAGNOSIS] Root cause: The PostgreSQL primary instance is experiencing a connection refusal (ECONNREFUSED) on port 5432 due
[DIAGNOSIS] Confidence: MEDIUM
[DIAGNOSIS] Latency   : 17.168s

Diagnosis test passed
Root cause: The PostgreSQL primary instance is experiencing a connection refusal (ECONNREFUSED) on port 5432 due to a network configuration issue.


## 4. Resolution Agent
Fetches runbook steps from knowledge base
Generates ordered fix procedure specific to FinTechFlow

In [7]:
RESOLUTION_PROMPT = PromptTemplate.from_template("""
You are a senior SRE at FinTechFlow creating a resolution plan.

INCIDENT:
{incident_text}

SEVERITY: {severity} | CATEGORY: {category}
ROOT CAUSE: {root_cause}

RELEVANT RUNBOOKS AND PROCEDURES:
{runbooks}

Create a concrete step-by-step resolution plan for FinTechFlow engineers.
Use specific commands and tools from our stack:
PostgreSQL, Redis, Kafka, EKS/kubectl, Vault, Prometheus, Grafana.

Respond in this exact format:
STEPS:
1. [specific action with exact command if applicable]
2. [specific action with exact command if applicable]
3. [specific action with exact command if applicable]
4. [verification step]
ESTIMATED_TIME: [X minutes]
ESCALATE_IF: [condition that requires escalation]
""")


def resolution_agent(state: AgentState) -> AgentState:
    """
    Agent 3: Resolution
    Input : diagnosis output
    Output: ordered resolution steps, time estimate
    Uses  : hybrid retrieval from knowledge_base + playbooks
    LLM   : YES — Qwen2.5 7B via Ollama
    """
    start = time.time()
    print(f"\n[RESOLUTION] Fetching runbooks...")

    try:
        # Build targeted query from diagnosis
        runbook_query = (
            f"{state['category']} {state['incident_text'][:150]} "
            f"resolution steps fix procedure"
        )

        # Search knowledge_base for runbooks + QA
        kb_results = retrieve(
            query            = runbook_query,
            collection_names = ['knowledge_base', 'playbooks'],
            top_k_fetch      = 10,
            top_k_return     = 4,
            use_reranking    = True
        )

        runbooks_text = "\n\n".join([
            f"[{i+1}] {r['text'][:400]}"
            for i, r in enumerate(kb_results)
        ]) or "No runbooks found — use standard procedure"

        state['retrieved_runbooks'] = [
            r['text'][:200] for r in kb_results
        ]

        print(f"[RESOLUTION] Retrieved {len(kb_results)} runbook entries")
        print(f"[RESOLUTION] Generating resolution plan...")

        prompt = RESOLUTION_PROMPT.format(
            incident_text = state['incident_text'],
            severity      = state['severity'],
            category      = state['category'],
            root_cause    = state['root_cause'],
            runbooks      = runbooks_text
        )

        response = llm.invoke(prompt)

        # Parse resolution steps
        steps        = []
        est_time     = 30
        in_steps     = False
        escalate_if  = ""

        for line in response.split('\n'):
            line = line.strip()
            if line == 'STEPS:':
                in_steps = True
                continue
            if line.startswith('ESTIMATED_TIME:'):
                in_steps  = False
                time_str  = line.replace('ESTIMATED_TIME:', '').strip()
                try:
                    est_time = int(''.join(
                        filter(str.isdigit, time_str)
                    ))
                except Exception:
                    est_time = 30
            elif line.startswith('ESCALATE_IF:'):
                escalate_if = line.replace('ESCALATE_IF:', '').strip()
            elif in_steps and line and line[0].isdigit():
                steps.append(line)

        state['resolution_steps']    = steps if steps else [
            "Check service logs",
            "Restart affected service",
            "Verify recovery"
        ]
        state['estimated_time_mins'] = est_time
        state['escalation_needed']   = state['severity'] == 'P1'

        latency = round(time.time() - start, 3)
        state['agent_latencies']['resolution'] = latency

        print(f"[RESOLUTION] Steps    : {len(state['resolution_steps'])}")
        print(f"[RESOLUTION] ETA      : {est_time} mins")
        print(f"[RESOLUTION] Escalate : {state['escalation_needed']}")
        print(f"[RESOLUTION] Latency  : {latency}s")

    except Exception as e:
        state['resolution_steps']    = ["Manual investigation required"]
        state['estimated_time_mins'] = 60
        state['error']               = f"Resolution error: {str(e)}"
        print(f"[RESOLUTION] Error: {e}")

    return state


print("Testing resolution agent...")
result_state = resolution_agent(result_state)
print(f"\nResolution test passed")
print(f"Steps: {len(result_state['resolution_steps'])}")
print(f"First step: {result_state['resolution_steps'][0] if result_state['resolution_steps'] else 'none'}")

Testing resolution agent...

[RESOLUTION] Fetching runbooks...


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[RESOLUTION] Retrieved 4 runbook entries
[RESOLUTION] Generating resolution plan...
[RESOLUTION] Steps    : 4
[RESOLUTION] ETA      : 30 mins
[RESOLUTION] Escalate : True
[RESOLUTION] Latency  : 38.791s

Resolution test passed
Steps: 4
First step: 1. **Check the current max_connections setting and identify the number of active connections.**


## 5. Communication Agent
Drafts stakeholder Slack message and preliminary postmortem
Tone calibrated to severity — P1 is urgent, P3 is informational

In [9]:
COMMS_PROMPT = PromptTemplate.from_template("""
You are the incident commander at FinTechFlow drafting communications.

INCIDENT SUMMARY:
- Text     : {incident_text}
- Severity : {severity}
- Category : {category}
- Root Cause: {root_cause}
- Resolution: {resolution_steps}
- ETA      : {eta} minutes

TASK 1: Write a Slack message for the #incidents channel.
Tone must match severity:
- P1: urgent, clear impact, immediate action
- P2: informative, impact scoped, action in progress
- P3: low urgency, minor impact, being monitored

TASK 2: Write a preliminary postmortem outline.

Respond in this exact format:

SLACK_MESSAGE:
[Your slack message here — 3-4 sentences max]

POSTMORTEM_OUTLINE:
Title: [incident title]
Severity: {severity}
Status: Investigating
Summary: [1-2 sentences]
Timeline:
- T+0: [initial alert]
- T+5: [investigation started]
- T+X: [resolution applied]
Root Cause: [root cause]
Action Items:
- [improvement 1]
- [improvement 2]
""")


def comms_agent(state: AgentState) -> AgentState:
    """
    Agent 4: Communication
    Input : full pipeline state
    Output: stakeholder message + postmortem draft
    LLM   : YES — higher temperature for better writing
    """
    start = time.time()
    print(f"\n[COMMS] Drafting stakeholder communications...")

    # Use higher temperature for communication
    comms_llm = Ollama(
        model       = "qwen2.5:7b",
        base_url    = "http://localhost:11434",
        temperature = 0.4,
        num_ctx     = 4096
    )

    try:
        steps_text = "\n".join(
            state.get('resolution_steps', [])
        )

        prompt = COMMS_PROMPT.format(
            incident_text    = state['incident_text'][:300],
            severity         = state['severity'],
            category         = state['category'],
            root_cause       = state['root_cause'],
            resolution_steps = steps_text[:400],
            eta              = state.get('estimated_time_mins', 30)
        )

        response = comms_llm.invoke(prompt)

        # Parse response
        slack_msg   = ""
        postmortem  = ""
        in_slack    = False
        in_pm       = False

        for line in response.split('\n'):
            if line.strip() == 'SLACK_MESSAGE:':
                in_slack = True
                in_pm    = False
                continue
            if line.strip() == 'POSTMORTEM_OUTLINE:':
                in_slack = False
                in_pm    = True
                continue
            if in_slack and line.strip():
                slack_msg += line + "\n"
            if in_pm and line.strip():
                postmortem += line + "\n"

        state['stakeholder_message'] = slack_msg.strip() or (
            f"[{state['severity']}] Incident in progress. "
            f"Category: {state['category']}. "
            f"Root cause: {state['root_cause'][:100]}. "
            f"ETA: {state.get('estimated_time_mins', 30)} mins."
        )

        state['postmortem_draft'] = postmortem.strip() or (
            f"Title: {state['category'].title()} Incident\n"
            f"Severity: {state['severity']}\n"
            f"Root Cause: {state['root_cause']}\n"
            f"Status: Investigating"
        )

        latency = round(time.time() - start, 3)
        state['agent_latencies']['comms'] = latency

        print(f"[COMMS] Slack message ready")
        print(f"[COMMS] Postmortem draft ready")
        print(f"[COMMS] Latency: {latency}s")

    except Exception as e:
        state['stakeholder_message'] = f"Incident in progress. Severity: {state['severity']}."
        state['postmortem_draft']    = "Postmortem draft pending."
        state['error']               = f"Comms error: {str(e)}"
        print(f"[COMMS] Error: {e}")

    return state


print("Testing comms agent...")
result_state = comms_agent(result_state)
print(f"\nComms test passed")
print(f"\nSlack message:\n{result_state['stakeholder_message']}")

Testing comms agent...

[COMMS] Drafting stakeholder communications...
[COMMS] Slack message ready
[COMMS] Postmortem draft ready
[COMMS] Latency: 49.517s

Comms test passed

Slack message:
PostgreSQL connection refused on payment-service (ECONNREFUSED) - max_connections exhausted. Immediate action required to resolve. ETA 30 minutes.


## 6. Wire LangGraph Pipeline
Connect all 4 agents into a state machine
Sequential flow: Triage → Diagnosis → Resolution → Comms

In [10]:
def build_pipeline() -> StateGraph:
    """
    Build the 4-agent LangGraph state machine.

    Flow:
    START → triage → diagnosis → resolution → comms → END
    """
    graph = StateGraph(AgentState)

    # Add all 4 agents as nodes
    graph.add_node("triage",     triage_agent)
    graph.add_node("diagnosis",  diagnosis_agent)
    graph.add_node("resolution", resolution_agent)
    graph.add_node("comms",      comms_agent)

    # Sequential edges
    graph.set_entry_point("triage")
    graph.add_edge("triage",     "diagnosis")
    graph.add_edge("diagnosis",  "resolution")
    graph.add_edge("resolution", "comms")
    graph.add_edge("comms",      END)

    return graph.compile()


pipeline = build_pipeline()
print("LangGraph pipeline compiled")
print("Flow: triage → diagnosis → resolution → comms → END")

LangGraph pipeline compiled
Flow: triage → diagnosis → resolution → comms → END


## 7. Full Pipeline Test
Run 3 complete incidents end to end
Covers: database (P1), security (P1), monitoring (P2)

In [11]:
def run_incident(incident_text: str) -> dict:
    """Run full 4-agent pipeline on an incident."""
    initial_state = AgentState(
        incident_text        = incident_text,
        severity             = "",
        category             = "",
        severity_confidence  = 0.0,
        category_confidence  = 0.0,
        severity_source      = "",
        retrieved_incidents  = [],
        retrieved_postmortems= [],
        root_cause           = "",
        diagnosis_confidence = 0.0,
        retrieved_runbooks   = [],
        resolution_steps     = [],
        estimated_time_mins  = 0,
        escalation_needed    = False,
        stakeholder_message  = "",
        postmortem_draft     = "",
        agent_latencies      = {},
        error                = None
    )

    start  = time.time()
    result = pipeline.invoke(initial_state)
    total  = round(time.time() - start, 3)
    result['agent_latencies']['total'] = total

    return result


# Test incidents covering key FTF scenarios
test_incidents = [
    {
        "name": "Database P1",
        "text": "ALERT: payment-service failing health checks. PostgreSQL connection refused on port 5432. Error: ECONNREFUSED max_connections=200 reached. 50,000 transactions pending. Team: payments-sre"
    },
    {
        "name": "Security P1",
        "text": "CRITICAL: HashiCorp Vault sealed unexpectedly during AWS KMS key rotation. All microservices unable to read secrets. payment-service, order-service, fraud-detection all failing. Team: security-sre"
    },
    {
        "name": "Monitoring P2",
        "text": "WARNING: PagerDuty alert storm detected. 400 alerts fired in last 5 minutes from payment-service. Alert Manager may be misconfigured. Real incidents may be masked. Team: platform-sre"
    }
]

print("=" * 60)
print("FULL PIPELINE TEST — 3 FINTECHFLOW INCIDENTS")
print("=" * 60)

results = []
for test in test_incidents:
    print(f"\n{'='*60}")
    print(f"INCIDENT: {test['name']}")
    print(f"{'='*60}")

    result = run_incident(test['text'])
    results.append(result)

    print(f"\nPIPELINE RESULTS:")
    print(f"  Severity  : {result['severity']} "
          f"({result['severity_confidence']:.2f}) "
          f"via {result['severity_source']}")
    print(f"  Category  : {result['category']} "
          f"({result['category_confidence']:.2f})")
    print(f"  Root cause: {result['root_cause'][:120]}")
    print(f"  Steps     : {len(result['resolution_steps'])}")
    for i, step in enumerate(result['resolution_steps'][:3], 1):
        print(f"    {i}. {step[:80]}")
    print(f"\n  Stakeholder message:")
    print(f"  {result['stakeholder_message'][:200]}")
    print(f"\n  Latencies:")
    for agent, lat in result['agent_latencies'].items():
        print(f"    {agent:12}: {lat}s")

FULL PIPELINE TEST — 3 FINTECHFLOW INCIDENTS

INCIDENT: Database P1

[TRIAGE] Processing incident...
[TRIAGE] Severity : P1 (0.85) via rule_based
[TRIAGE] Category : database (0.98)
[TRIAGE] Latency  : 0.192s

[DIAGNOSIS] Retrieving similar incidents...
[DIAGNOSIS] Retrieved 3 incidents, 3 postmortems
[DIAGNOSIS] Generating diagnosis with LLM...
[DIAGNOSIS] Root cause: The PostgreSQL instance is overloaded, reaching the max_connections limit of 200, causing connection
[DIAGNOSIS] Confidence: HIGH
[DIAGNOSIS] Latency   : 7.635s

[RESOLUTION] Fetching runbooks...
[RESOLUTION] Retrieved 4 runbook entries
[RESOLUTION] Generating resolution plan...
[RESOLUTION] Steps    : 4
[RESOLUTION] ETA      : 30 mins
[RESOLUTION] Escalate : True
[RESOLUTION] Latency  : 23.823s

[COMMS] Drafting stakeholder communications...
[COMMS] Slack message ready
[COMMS] Postmortem draft ready
[COMMS] Latency: 13.292s

PIPELINE RESULTS:
  Severity  : P1 (0.85) via rule_based
  Category  : database (0.98)
  Root ca

## 8. Save Pipeline Module
Export pipeline as reusable module for FastAPI

In [12]:
pipeline_code = '''"""
TechOps Intelligence Platform — Agent Pipeline
FinTechFlow incident intelligence system

Usage:
    from src.agents.pipeline import run_incident

    result = run_incident(
        "PostgreSQL connection refused port 5432"
    )
"""
import os
import sys
import time
from pathlib import Path
from typing import TypedDict, Optional, List

os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

PROJECT_ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(PROJECT_ROOT))

from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph, END

from src.agents.triage_classifier import classify_incident
from src.retrieval.retriever import retrieve


class AgentState(TypedDict):
    incident_text        : str
    severity             : str
    category             : str
    severity_confidence  : float
    category_confidence  : float
    severity_source      : str
    retrieved_incidents  : List[str]
    retrieved_postmortems: List[str]
    root_cause           : str
    diagnosis_confidence : float
    retrieved_runbooks   : List[str]
    resolution_steps     : List[str]
    estimated_time_mins  : int
    escalation_needed    : bool
    stakeholder_message  : str
    postmortem_draft     : str
    agent_latencies      : dict
    error                : Optional[str]


_llm = None
def get_llm(temperature=0.1):
    return Ollama(
        model       = "qwen2.5:7b-instruct-q4_K_M",
        base_url    = "http://localhost:11434",
        temperature = temperature,
        num_ctx     = 4096
    )


def triage_node(state):
    start  = time.time()
    result = classify_incident(state["incident_text"])
    state.update({
        "severity"           : result["severity"],
        "category"           : result["category"],
        "severity_confidence": result["severity_confidence"],
        "category_confidence": result["category_confidence"],
        "severity_source"    : result.get("severity_source", "distilbert"),
    })
    state["agent_latencies"]["triage"] = round(time.time()-start, 3)
    return state


DIAGNOSIS_PROMPT = PromptTemplate.from_template("""
You are a senior SRE at FinTechFlow.
INCIDENT: {incident_text}
SEVERITY: {severity} | CATEGORY: {category}
CONTEXT: {context}
ROOT_CAUSE: [one sentence]
CONFIDENCE: [HIGH/MEDIUM/LOW]
IMMEDIATE_ACTION: [first step]
""")


def diagnosis_node(state):
    start    = time.time()
    llm      = get_llm(0.1)
    incidents = retrieve(state["incident_text"],
                         ["incidents"], top_k_return=3)
    pms       = retrieve(state["incident_text"],
                         ["postmortems"], top_k_return=3)
    context   = "\\n".join([r["text"][:300]
                            for r in incidents + pms])
    response  = llm.invoke(DIAGNOSIS_PROMPT.format(
        incident_text=state["incident_text"],
        severity=state["severity"],
        category=state["category"],
        context=context[:1500]
    ))
    root_cause = "Root cause under investigation"
    for line in response.split("\\n"):
        if line.startswith("ROOT_CAUSE:"):
            root_cause = line.replace("ROOT_CAUSE:", "").strip()
    state.update({
        "root_cause"           : root_cause,
        "diagnosis_confidence" : 0.8,
        "retrieved_incidents"  : [r["text"][:200] for r in incidents],
        "retrieved_postmortems": [r["text"][:200] for r in pms],
    })
    state["agent_latencies"]["diagnosis"] = round(time.time()-start, 3)
    return state


RESOLUTION_PROMPT = PromptTemplate.from_template("""
Senior SRE at FinTechFlow. Create resolution plan.
INCIDENT: {incident_text}
ROOT CAUSE: {root_cause}
RUNBOOKS: {runbooks}
STEPS:
1. [action]
2. [action]
3. [action]
4. [verify]
ESTIMATED_TIME: [X minutes]
""")


def resolution_node(state):
    start    = time.time()
    llm      = get_llm(0.1)
    runbooks = retrieve(
        f"{state['category']} {state['incident_text'][:150]}",
        ["knowledge_base", "playbooks"], top_k_return=3
    )
    rb_text  = "\\n".join([r["text"][:300] for r in runbooks])
    response = llm.invoke(RESOLUTION_PROMPT.format(
        incident_text=state["incident_text"][:200],
        root_cause=state["root_cause"],
        runbooks=rb_text[:1000]
    ))
    steps    = []
    est_time = 30
    in_steps = False
    for line in response.split("\\n"):
        line = line.strip()
        if line == "STEPS:":
            in_steps = True
        elif line.startswith("ESTIMATED_TIME:"):
            in_steps = False
            try:
                est_time = int("".join(filter(str.isdigit, line)))
            except Exception:
                est_time = 30
        elif in_steps and line and line[0].isdigit():
            steps.append(line)
    state.update({
        "resolution_steps"   : steps or ["Check logs", "Restart service", "Verify"],
        "estimated_time_mins": est_time,
        "escalation_needed"  : state["severity"] == "P1",
        "retrieved_runbooks" : [r["text"][:200] for r in runbooks],
    })
    state["agent_latencies"]["resolution"] = round(time.time()-start, 3)
    return state


COMMS_PROMPT = PromptTemplate.from_template("""
Incident commander at FinTechFlow.
INCIDENT: {incident_text}
SEVERITY: {severity} | ROOT CAUSE: {root_cause}
ETA: {eta} minutes
SLACK_MESSAGE:
[3-4 sentence update for #incidents channel]
POSTMORTEM_OUTLINE:
Title: [title]
Summary: [summary]
""")


def comms_node(state):
    start    = time.time()
    llm      = get_llm(0.4)
    response = llm.invoke(COMMS_PROMPT.format(
        incident_text=state["incident_text"][:200],
        severity=state["severity"],
        root_cause=state["root_cause"],
        eta=state.get("estimated_time_mins", 30)
    ))
    slack = ""
    pm    = ""
    in_s  = False
    in_p  = False
    for line in response.split("\\n"):
        if "SLACK_MESSAGE:" in line:
            in_s = True; in_p = False; continue
        if "POSTMORTEM_OUTLINE:" in line:
            in_s = False; in_p = True; continue
        if in_s and line.strip():
            slack += line + "\\n"
        if in_p and line.strip():
            pm += line + "\\n"
    state.update({
        "stakeholder_message": slack.strip() or f"[{state['severity']}] Incident in progress.",
        "postmortem_draft"   : pm.strip() or f"Title: {state['category']} incident\\nSeverity: {state['severity']}",
    })
    state["agent_latencies"]["comms"] = round(time.time()-start, 3)
    return state


def build_pipeline():
    g = StateGraph(AgentState)
    g.add_node("triage",     triage_node)
    g.add_node("diagnosis",  diagnosis_node)
    g.add_node("resolution", resolution_node)
    g.add_node("comms",      comms_node)
    g.set_entry_point("triage")
    g.add_edge("triage",     "diagnosis")
    g.add_edge("diagnosis",  "resolution")
    g.add_edge("resolution", "comms")
    g.add_edge("comms",      END)
    return g.compile()


_pipeline = None

def run_incident(incident_text: str) -> dict:
    global _pipeline
    if _pipeline is None:
        _pipeline = build_pipeline()

    state = AgentState(
        incident_text        = incident_text,
        severity             = "",
        category             = "",
        severity_confidence  = 0.0,
        category_confidence  = 0.0,
        severity_source      = "",
        retrieved_incidents  = [],
        retrieved_postmortems= [],
        root_cause           = "",
        diagnosis_confidence = 0.0,
        retrieved_runbooks   = [],
        resolution_steps     = [],
        estimated_time_mins  = 0,
        escalation_needed    = False,
        stakeholder_message  = "",
        postmortem_draft     = "",
        agent_latencies      = {},
        error                = None
    )
    start              = time.time()
    result             = _pipeline.invoke(state)
    result["agent_latencies"]["total"] = round(time.time()-start, 3)
    return result
'''

pipeline_path = PROJECT_ROOT / "src/agents/pipeline.py"
pipeline_path.parent.mkdir(parents=True, exist_ok=True)
with open(pipeline_path, 'w', encoding='utf-8') as f:
    f.write(pipeline_code)

print(f"Pipeline module saved: {pipeline_path}")
print("\nUsage:")
print("  from src.agents.pipeline import run_incident")
print("  result = run_incident('PostgreSQL connection refused')")

Pipeline module saved: C:\Users\sudha\techops-intelligence\src\agents\pipeline.py

Usage:
  from src.agents.pipeline import run_incident
  result = run_incident('PostgreSQL connection refused')


In [14]:
print("=" * 60)
print("NOTEBOOK 08 - AGENT PIPELINE COMPLETE")
print("=" * 60)

print("\nAgents built:")
print("  1. Triage     - hybrid DistilBERT + rules")
print("  2. Diagnosis  - RAG retrieval + Qwen2.5")
print("  3. Resolution - runbook retrieval + Qwen2.5")
print("  4. Comms      - stakeholder message + postmortem")

print("\nPipeline latencies (last run):")
if results:
    for agent, lat in results[-1]['agent_latencies'].items():
        print(f"  {agent:12}: {lat}s")

print("\nPipeline module: src/agents/pipeline.py")
print("=" * 60)

NOTEBOOK 08 - AGENT PIPELINE COMPLETE

Agents built:
  1. Triage     - hybrid DistilBERT + rules
  2. Diagnosis  - RAG retrieval + Qwen2.5
  3. Resolution - runbook retrieval + Qwen2.5
  4. Comms      - stakeholder message + postmortem

Pipeline latencies (last run):
  triage      : 0.107s
  diagnosis   : 7.249s
  resolution  : 21.047s
  comms       : 13.037s
  total       : 41.444s

Pipeline module: src/agents/pipeline.py
